In [1]:
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

MODEL = "gemma4:31b-cloud"

# smoke test
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "hello"}]
)
print(response.choices[0].message.content)

Hello! How can I help you today?


In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
print(len(files))

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

72


In [4]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

for r in results:
    print(r["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


In [5]:
import importlib
import rag_helper
importlib.reload(rag_helper)
from rag_helper import RAGBase

rag = RAGBase(index=index, llm_client=client, model=MODEL)

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = rag.rag(query)

print(answer)
print(usage)
print("Input tokens:", usage.prompt_tokens)

The agentic loop keeps calling the model by wrapping the interaction in a `while True` loop that continues until a specific exit condition is met. Here is the step-by-step process based on the provided context:

1.  **Call the Model:** The loop sends the current message history (including instructions, the user question, and any previous tool results) to the model.
2.  **Check for Function Calls:** The code iterates through the model's response output. It uses a flag (e.g., `has_function_calls = False`) to track if the model has requested any tool use.
3.  **Execute Tools:** If the model returns a `function_call` item, the loop:
    *   Sets `has_function_calls = True`.
    *   Runs the requested function (e.g., `search`) using a helper.
    *   Appends the result of that function back into the message history.
4.  **Repeat or Exit:** 
    *   **If `has_function_calls` is True:** The loop continues to the next iteration, sending the updated history (now containing the tool output) back

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


In [7]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

rag_chunked = RAGBase(index=chunk_index, llm_client=client, model=MODEL)

query = "How does the agentic loop keep calling the model until it stops?"
answer_chunked, usage_chunked = rag_chunked.rag(query)

print(answer_chunked)
print(usage_chunked)
print("Input tokens (chunked):", usage_chunked.prompt_tokens)

The agentic loop uses a `while True` loop that continues to call the model until the model returns a response that contains no function calls.

Specifically, the process works as follows:
1. A flag (e.g., `has_function_calls`) is set to `False` at the start of each iteration.
2. The model is called, and its output is processed.
3. If the model's output includes a `function_call`, the code executes that tool, appends the result to the message history, and sets the `has_function_calls` flag to `True`.
4. If the flag remains `False` at the end of the iteration (meaning the model returned a final answer without requesting any further tool calls), the `break` command is triggered and the loop stops.
CompletionUsage(completion_tokens=172, prompt_tokens=2614, total_tokens=2786, completion_tokens_details=None, prompt_tokens_details=None)
Input tokens (chunked): 2614


In [8]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
from toyaikit.chat import IPythonChatInterface


class SearchTools:
    def __init__(self, index):
        self.index = index
        self.call_count = 0  # track how many times search is called

    def search(self, query: str):
        """
        Search the course lesson pages for content matching the given query.

        Args:
            query (str): Search query text to look up in the lesson pages.

        Returns:
            List[Dict[str, Any]]: A list of matching lesson chunks.
        """
        self.call_count += 1
        print(f"[search call #{self.call_count}] query: {query}")
        results = self.index.search(query, num_results=5)
        return results


search_tools = SearchTools(chunk_index)

tools = Tools()
tools.add_tools(search_tools)

llm_client = OpenAIChatCompletionsClient(
    model=MODEL,
    client=client
)

developer_prompt = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
"""

runner = OpenAIChatCompletionsRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=IPythonChatInterface(),
    llm_client=llm_client
)

result = runner.loop(prompt="How does the agentic loop work, and how is it different from plain RAG?")

print(result.last_message)
print("Number of search calls:", search_tools.call_count)

[search call #1] query: agentic loop
[search call #2] query: agentic loop vs RAG
[search call #3] query: what is an agentic loop
The **agentic loop** is a design pattern that places the LLM "in the driver's seat," allowing it to iteratively decide which actions to take until a goal is achieved.

### How the Agentic Loop Works
Instead of a linear sequence of steps, an agentic loop functions as a `while` loop that continues until the model determines it has a final answer. The process generally follows these steps:

1.  **Call the LLM:** The model is provided with a goal (prompt), instructions (developer prompt), and a history of previous actions (memory).
2.  **Analyze & Decide:** The LLM decides if it needs more information. If so, it returns a **tool call** (e.g., a request to use a `search` function).
3.  **Execute Tool:** The system executes the requested tool and captures the output.
4.  **Update Memory:** The tool's result is appended to the conversation history.
5.  **Repeat:** T

/Users/cr7/Documents/code/llm-zoomcamp-2026-code/01-agentic-rag/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:542: UnknownModelWarning: No pricing data for model '31b-cloud'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(
